# Scrapers for MyDramaList

You will follow the instructions in Part 4 of Week 2 Tasks. In the top part of the notebook summarize through a table of content what you decided to do and then explain why.

**Author**: Crystal Zhao    
**Date**: 9/13.    

I decided to attempt Route A, which deals with combining paginated aggregation pages and dedicated pages. My research question will be focused on the top rated movies (5000 results): What is the national composition of top rated movies?   

This would require me to look at this aggregated page: https://mydramalist.com/movies/top. Upon inspection, I would be interested in this element:

"""
<span class="text-muted">Korean Movie - 2017</span>
"""

Rationale/pseudocode: I would go through all pages of this aggregated webpage (250 pages...), and for each page, save the 1) name, 2) rank, and 3) nationality of each movie. For nationality, I would split up the String by space and only extract the first word.

**Table of Contents**

1. [Scraping Aggregated Pages](#sec1)
2. [Scraping Across Pages](#sec2)
3. [Scraping Dedicated Pages](#sec3)
4. [Scraping by Scrolling](#sec4)

<a id="sec1"></a>

## Scraping Aggregated Pages

### a. Identifying the Movie Cards

Each movie card is a .box element,   
Each .box contains a .row element, with:  

1. ".ranking.pull-right"
2. ".text-primary-title" class with a hyperlink that directly leads to a dedicated page through a href (we should also save this so we can navigate to dedicated pages when necessary)
3. followed by the actual title
3. and finally ".text-muted" (e.g., Chinese Movie - 2019).

In [2]:
from seleniumbase import Driver
from selenium.webdriver.common.by import By
import re
import requests

url = "https://mydramalist.com/movies/top?page=2"

In [3]:
def fetch_page_content(url):
    """Fetches HTML content from a URL and checks status code."""
    response = requests.get(url) # call the function get with the URL, response is a Python object with a lot of attribues

    print(f"URL: {url}")
    print(f"Status Code: {response.status_code}")

    if response.status_code == 200: # check the status code to make sure we got the page from the server 
        return response.text # text is an attribute 
    else:
        print(f"Failed to fetch page. Status code: {response.status_code}")
        return None

In [6]:
page = fetch_page_content(url)

URL: https://mydramalist.com/movies/top?page=2
Status Code: 200


Now that we have the static page html, we can try to parse it with Selenium first to make sure we are looking in the right places for the information we want.

In [15]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(page, "html.parser")
top_movies = soup.find('div', class_='col-lg-8 col-md-8') # have to first narrow down to this so we ar enot including other boxes 
movies = top_movies.find_all("div", class_="box")
print(len(movies))

20


The tricky part for me right now is getting both the title and the link, so I am gonna investigate.

In [31]:
movie_1 = movies[0]
tags = movie_1.find_all('a')
print(len(tags)) 
print(tags)
tags[0]

3
[<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>, <a href="/24153-be-with-you">Be with You</a>, <a class="btn simple btn-manage-list" data-id="24153" data-stats="mylist:24153" rel="nofollow"><span><i class="far fa-plus"></i></span></a>]


<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>

What we need is the first a tag that is a block with the link, so we only need to do _find but let's verify that:

In [32]:
title_el = movie_1.find('a')
print(title_el)

<a class="block" href="/24153-be-with-you">
<img alt="Be with You" class="img-responsive cover lazy" data-src="https://i.mydramalist.com/pnvlns.jpg?v=1"/>
</a>


In [33]:
title_el.text

'\n\n'

We can't just call text, so we need to get the alternative description under image.

In [36]:
img_tag = movie_1.find('img')
img_tag['alt']

'Be with You'

In [ ]:
movie_data = []
for m in movies:
        movie_dict = {} 
        
        movie_dict['rank'] = m.find(class_='ranking pull-right').get_text()

        title_tag = movie_1.find('img')
        movie_dict['title'] = title_tag['alt']

        link_el = m.find('a')
        movie_dict['link'] = link_el['href']
        
        if movie_dict:
            movie_data.append(movie_dict)

In [38]:
movie_data[0]

{'rank': '#21', 'title': 'Be with You', 'link': '/24153-be-with-you'}

Now, to make sure we are also getting the country from text-muted.

In [26]:
text = 'Korean Movie - 2018'
text.split(' ')[0].strip()

'Korean'

Success! Now we should be able to wrap this in a function but in Selenium format.

### b. Defining the 'parse_movies' function

We would call this function after we get to each page, so we would run this function for each page of information that we get.

With this in mind, let's try to scrape just the first page (all movies) and save the information in 1 list.

In [27]:
from seleniumbase import Driver

def parse_movie(boxes):
    """
    Given a bunch of boxes on one page, extracts information from the entire movie through each box .
    """
    if not boxes:
        return []

    page_movies_data = []
    
    for movie in boxes:
        movie_dict = {}

        movie_dict['rank'] = movie.find_element(By.CSS_SELECTOR, '.ranking.pull-right').text
        title_tag = movie.find_element(By.TAG_NAME, 'img')
        movie_dict['title'] = title_tag.get_attribute('alt')
        
        link_el = movie.find_element(By.TAG_NAME,'a')
        movie_dict['link'] = link_el.get_attribute('href')

        description = movie.find_element(By.CSS_SELECTOR, '.text-muted').text
        movie_dict['country'] = description.split(' ')[0].strip()
        
        if movie_dict:
            page_movies_data.append(movie_dict)

    return page_movies_data

Now the challenge is to figure out how Selenium can get the "box" elements I need.



### c. Getting the Boxes using Selenium

In [12]:
from seleniumbase import Driver
from selenium.webdriver.common.by import By

url = "https://mydramalist.com/movies/top?page=2"

with Driver(browser='chrome', page_load_strategy='eager') as driver:
    driver.get(url)
    parent = driver.find_element(By.CSS_SELECTOR, '.col-lg-8.col-md-8')
    print("hello")
    boxes = parent.find_elements(By.CSS_SELECTOR,'.box')
    print(len(boxes))

hello
20


This means we got all 20 movies on this page. Now, let's call parse_movie on these boxes for page 1 first.

In [28]:
with Driver(browser='chrome', page_load_strategy='eager') as driver:
    driver.get(url)
    parent = driver.find_element(By.CSS_SELECTOR, '.col-lg-8.col-md-8')
    boxes = parent.find_elements(By.CSS_SELECTOR,'.box')
    print(f"Scraping {len(boxes)} boxes")
    page_movie_data = parse_movie(boxes) 


Scraping 20 boxes


In [29]:
page_movie_data[:2]

[{'rank': '#21',
  'title': 'Be with You',
  'link': 'https://mydramalist.com/24153-be-with-you',
  'country': 'Korean'},
 {'rank': '#22',
  'title': 'Along with the Gods 2: The Last 49 Days',
  'link': 'https://mydramalist.com/28305-along-with-the-gods-the-last-49-days',
  'country': 'Korean'}]

<a id="sec2"></a>

## 2. Scraping Across Pages

In [1]:
import math

url = "https://mydramalist.com/movies/top"

with Driver(browser='chrome', page_load_strategy='eager') as driver:
    driver.open(url)

    # 1. Extract total results count (e.g., "5000 results" -> 105)
    total_text = driver.get_text(".m-b-sm.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])
    print(total_results)

    """
    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    all_data = []
    # 4. Loop through each page URL, but just 5 for now to make sure it works
    for page in range(1, 6):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 5. Extract items on the current page
        boxes = driver.find_elements(".box") # this is a list of cards as WebElements
        page_data = parse_movie(boxes)
        all_data.append(page_data)
        print(f"Page {page}: Scraped {len(boxes)} movies")

    """

NameError: name 'Driver' is not defined

I'm running into a lot of trouble with how long it takes the code to run... I also got a ReadTimeoutError: HTTPConnectionPool(host='localhost', port=52864): Read timed out. (read timeout=120) error.

In [ ]:
import math
from seleniumbase import Driver

url = "https://mydramalist.com/movies/top"

with Driver() as driver:
    driver.open(url)

    # 1. Extract total results count (e.g., "5000 results" -> 5000)
    total_text = driver.get_text(".m-b-sm.pull-right") # This will be a string value
    total_results = int(total_text.split()[0])

    # 2. Calculate total pages (20 items per page)
    total_pages = math.ceil(total_results / 20)
    print(f"Total results: {total_results} | Total pages: {total_pages}")

    # 3. Loop through each page URL, but since this is just for demonstration, I will make the end limit 5
    """
    for page in range(1, 5):
        driver.open(f"{url}?page={page}")
        driver.sleep(1.0)

        # 4. Extract items on the current page
        cards = driver.find_elements(".box") # this is a list of cards as WebElements # the . means that the class is box; with a # means ID, with no dot or hash means HTML tag  
        print(f"Page {page}: Scraped {len(cards)} dramas")
    """

KeyboardInterrupt: 

<a id="sec3"></a>

## 3. Scraping Dedicated Pages

Scraping the top rated movie:

**If we only wanted to stay at the top**:
movie['title'] = soup.find('div', class='film-title) but split at "(" and only keep the first item in the split list 

movie['year'] = soup.find('film-subtitle text-sm') but split at "space" and only keep the last item 

The ones below all use class = 'hfs'
movie['rating'] = soup.find(id, show-detailsxx) --> <div class="hfs" itempropx="aggregateRating" Ratings: <b style="font-weight:bold;" --> itempropx="ratingValue">9.2</b>/10 from 28,712 users 

movie['#_watchers'] = we only want to keep the number portion of this code (<div class="hfs"># of Watchers: <b>104,422</b></div>), so split at : and strip() the second.

movie['#_reviewers] = we only want to keep the text-primary portion of this (<div class="hfs">Reviews: <a class="text-primary" href="/30499-the-youthful-you-who-was-so-beautiful/reviews">134 users</a></div>)

movie['synopsis'] = find class='show-synopsis', the first <p> tag

**Directly scraping details**:   
The below are all uder class = show-detailsxss:
li class - all p-a-0
movie['Native Title'].   
movie['Director].   
movie['Screenwriter'].   
movie['Genres'].   
movie['Tags'].   
movie['Coutry'].   
movie['release date'].   
movie['duration'].   
movie['score'].   
movie['ranked'].   
movie['popularity'].    
movie['content rating'].   

I will use BeautifulSoup to scrape one dedicated page.

In [5]:
from bs4 import BeautifulSoup

In [6]:
url = 'https://mydramalist.com/30499-the-youthful-you-who-was-so-beautiful'
page = fetch_page_content(url)
soup = BeautifulSoup(page, 'html.parser')

URL: https://mydramalist.com/30499-the-youthful-you-who-was-so-beautiful
Status Code: 200


In [10]:
details = soup.find(class_='show-detailsxss')
details

<div class="show-detailsxss"> <ul class="list m-a-0"> <li class="list-item p-a-0 m-b-sm related-content"> <b class="inline">Related Content</b> <div class="title"> <a class="text-primary" href="/763893-ru-ci-mei-li" title="Ru Ci Mei Li">Ru Ci Mei Li</a>  (Chinese adaptation)  </div> </li> <li class="list-item p-a-0"><b class="inline">Native Title:</b> <a href="/30499-the-youthful-you-who-was-so-beautiful" title="少年的你">少年的你</a></li> <li class="list-item p-a-0"><b class="inline">Also Known As:</b> <span class="mdl-aka-titles">  Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天   </span> </li> <li class="list-item p-a-0"><b class="inline">Director:</b> <a class="text-primary" href="/people/30541-derek-tsang">Derek Tsang</a> </li> <li class="list-item p-a-0"><b class="inline">Screenwriter:</b> <a class="text-primary" href="/people/65671-lam-wing-sam">Lam WIng Sam</a>,

The li tag means list item, and must always stand within a parent list item.

In [ ]:
all_items = details.find_all(class_='list-item p-a-0')
print(len(all_items))


14


<li class="list-item p-a-0" style="display:none;"><b class="inline">Watchers:</b> 104,422</li>

This is too disorganized! I'm gonna get the parent element instead.

In [34]:
specifics = details.find(class_='list m-a-0') # NOT find_all, because that gives you a list instead of a tag
type(specifics)

bs4.element.Tag

In [35]:
temp = []
for item in specifics.find_all('li'): # iterating through each list item
    text = item.text.split(':')[-1].strip()
    print(text)
    temp.append(text)


Related Content  Ru Ci Mei Li  (Chinese adaptation)
少年的你
Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天
Derek Tsang
Lam WIng Sam,  Li Yuan,  Xu Yi Meng
Psychological,  Youth,  Drama
School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)


In [36]:
temp

['Related Content  Ru Ci Mei Li  (Chinese adaptation)',
 '少年的你',
 'Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天',
 'Derek Tsang',
 'Lam WIng Sam,  Li Yuan,  Xu Yi Meng',
 'Psychological,  Youth,  Drama',
 'School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)']

Good! The rest are in the second list called hidden-md-up

In [37]:
more_items = details.find(class_='list m-a-0 hidden-md-up')
more_items

<ul class="list m-a-0 hidden-md-up"> <li class="list-item p-a-0"><b class="inline">Country:</b> China <i class="flag flags-c2"></i></li> <li class="list-item p-a-0"><b class="inline">Type:</b> Movie</li> <li class="list-item p-a-0"><b class="inline">Release Date:</b> Oct 25, 2019</li> <li class="list-item p-a-0"><b class="inline">Duration:</b> 2 hr. 18 min.</li> <li class="list-item p-a-0"><b class="inline">Score:</b> 9.2 <span class="hft">(scored by <a href="/30499-the-youthful-you-who-was-so-beautiful/statistics">28,712 users</a>)</span></li> <li class="list-item p-a-0"><b class="inline">Ranked:</b> #25</li> <li class="list-item p-a-0"><b class="inline">Popularity:</b> #127</li> <li class="list-item p-a-0"><b class="inline">Content Rating:</b> 15+ - Teens 15 or older</li> <li class="list-item p-a-0" style="display:none;"><b class="inline">Watchers:</b> 104,422</li> <li class="list-item p-a-0" style="display:none;"><b class="inline">Favorites:</b> 0</li> </ul>

In [ ]:
temp = []
for item in more_items.find_all('li'): # iterating through each list item
    text = item.text.split(':')[-1].strip()
    print(text)
    temp.append(text)

China
Movie
Oct 25, 2019
2 hr. 18 min.
9.2 (scored by 28,712 users)
#25
#127
15+ - Teens 15 or older
104,422
0


The last item, 0 refers to "favorites" but it does not appear on the webpage itself, so I am thinking of just dropping it.

In [48]:
top_movie = {}

title_tag = soup.find(class_='film-title')
top_movie['title'] = title_tag.text

details = soup.find(class_='show-detailsxss')
specifics = details.find(class_='list m-a-0') 
for item in specifics.find_all('li'): # iterating through each list item
    key = item.text.split(':')[0].strip()
    value = item.text.split(':')[-1].strip()
    top_movie[f"{key}"] = value
    print(value)

specifics_2 = details.find(class_='list m-a-0 hidden-md-up')
for item in specifics_2.find_all('li'):
    key = item.text.split(':')[0].strip()
    value = item.text.split(':')[-1].strip()
    top_movie[f"{key}"] = value
    print(value)

Related Content  Ru Ci Mei Li  (Chinese adaptation)
少年的你
Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天
Derek Tsang
Lam WIng Sam,  Li Yuan,  Xu Yi Meng
Psychological,  Youth,  Drama
School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)
China
Movie
Oct 25, 2019
2 hr. 18 min.
9.2 (scored by 28,712 users)
#25
#127
15+ - Teens 15 or older
104,422
0


In [49]:
top_movie

{'title': 'Better Days (2019)',
 'Related Content  Ru Ci Mei Li  (Chinese adaptation)': 'Related Content  Ru Ci Mei Li  (Chinese adaptation)',
 'Native Title': '少年的你',
 'Also Known As': 'Geng Hao De Ming Tian ,  In His Youth ,  Shao Nian De Ni ,  Shao Nian De Ni, Ru Ci Mei Li ,  The Youthful You ,  The Youthful You Who Was So Beautiful ,  少年的你, 如此美丽 ,  更好的明天',
 'Director': 'Derek Tsang',
 'Screenwriter': 'Lam WIng Sam,  Li Yuan,  Xu Yi Meng',
 'Genres': 'Psychological,  Youth,  Drama',
 'Tags': 'School Bullying, Coming Of Age, Social Commentary, High School, Teenager Female Lead, Poor Male Lead, Orphan Male Lead, Poor Female Lead, Smart Female Lead, Unusual Friendship (Vote tags)',
 'Country': 'China',
 'Type': 'Movie',
 'Release Date': 'Oct 25, 2019',
 'Duration': '2 hr. 18 min.',
 'Score': '9.2 (scored by 28,712 users)',
 'Ranked': '#25',
 'Popularity': '#127',
 'Content Rating': '15+ - Teens 15 or older',
 'Watchers': '104,422',
 'Favorites': '0'}

<a id="sec4"></a>

## 4. Scraping By Scrolling

This is unfortunately not possible with BeautifulSoup and I couldn't get Selenium to work on my computer.

In [ ]:
from seleniumbase import Driver

url = "https://mydramalist.com/movies/top"

with Driver() as driver:
    driver.open(url)
    driver.sleep(1)
    rendered_html = driver.get_page_source()

Python(31580) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(31581) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(31582) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(31588) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
Python(31589) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


In [ ]:
from seleniumbase import Driver

url = "https://cs.wellesley.edu/~cs315/scraping/infinite_dramas.html"

last_count = 0 # keep track of how many cards we have seen so far 

with Driver() as driver:
    driver.open(url)

    while True:
        # 1. Scroll to the bottom of the page using JavaScript
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);") # scrolls one card at a time 
        
        # 2. Wait for dynamic JavaScript content to render
        driver.sleep(1.0)
        
        # 3. Check the new count of target elements
        current_count = len(driver.find_elements(".drama-card"))
        print(f"Drama count: {current_count}")
        
        # 4. Exit the loop if no new items loaded after scrolling
        if current_count == last_count:
            print("Reached the bottom of the page.")
            full_html_content = driver.get_page_source() # adding this to get all content
            break
    
        last_count = current_count
